# Glaucoma Segmentation Comparison Lab (Drishti-GS Edition)

This notebook trains and compares segmentation models for **Optic Disc extraction** using the **Drishti-GS** dataset.

### Models to Compare:
1.  **Standard U-Net:** The baseline.
2.  **Attention U-Net:** The proposed hybrid method.
3.  **DeepLabV3:** The heavy-duty competitor.

### Metric:
**Dice Coefficient (F1 Score)**. Aim for > 0.90 on Drishti-GS.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd

# Check Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {DEVICE}")

## 1. Configuration
Update the paths below. Drishti-GS usually splits data into 'Training' and 'Test' folders with different mask folder names ('GT' vs 'Test_GT').

In [ ]:
# --- EXPERIMENT SETTINGS ---
SEG_MODEL_NAME = "AttentionUNet"   # Options: "UNet", "AttentionUNet", "DeepLabV3"

# TRAIN DATASET PATHS
TRAIN_IMG_DIR = "path/to/Drishti-GS1_files/Training/Images"
TRAIN_MASK_DIR = "path/to/Drishti-GS1_files/Training/GT"

# TEST DATASET PATHS
TEST_IMG_DIR = "path/to/Drishti-GS1_files/Test/Images"
TEST_MASK_DIR = "path/to/Drishti-GS1_files/Test/Test_GT" # Drishti often names the test mask folder 'Test_GT'
# ---------------------------

BATCH_SIZE = 4          # 4 is safe for 4GB VRAM
LEARNING_RATE = 1e-4
EPOCHS = 20
IMAGE_SIZE = 256        # Drishti images are large, resizing to 256 is standard

## 2. Drishti-GS Dataset Loader (Full Recursive Support)
**Update:** Now recursively scans folders to find files hidden in subfolders (e.g., 'Glaucoma/').

In [ ]:
class DrishtiDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Check if directories exist
        if not os.path.exists(image_dir):
            raise FileNotFoundError(f"❌ Image Directory not found: {image_dir}")
        # Allow mask_dir to be None for inference only
        self.has_masks = mask_dir is not None and os.path.exists(mask_dir)

        # --- AUTO-SCANNER for Images (Recursive) ---
        self.image_paths = []
        print(f"🔍 Scanning '{image_dir}' for images (recursive)...")
        for root, dirs, files in os.walk(image_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(root, file))
        print(f"✅ Found {len(self.image_paths)} images.")

        # --- AUTO-SCANNER for Masks (Recursive) ---
        self.mask_map = {}
        if self.has_masks:
            print(f"🔍 Scanning '{mask_dir}' for masks (recursive)...")
            for root, dirs, files in os.walk(mask_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg')):
                        # Map filename to its full absolute path
                        self.mask_map[file] = os.path.join(root, file)
            print(f"✅ Found {len(self.mask_map)} potential mask files.")
        else:
            print("⚠️ No mask directory provided. Running in Inference Mode (Images Only).")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_full_path = self.image_paths[index]
        img_name = os.path.basename(img_full_path)
        base_name = os.path.splitext(img_name)[0]
        
        # Strategy: Look for specific Drishti suffixes in our scanned map
        mask_path = None
        if self.has_masks:
            # Priority list of suffixes (Drishti uses these commonly)
            target_names = [
                f"{base_name}_ODsegSoftmap.png", 
                f"{base_name}_OD.png", 
                f"{base_name}.png"
            ]
            
            # Check exact matches
            for target in target_names:
                if target in self.mask_map:
                    mask_path = self.mask_map[target]
                    break
            
            # Fallback: Fuzzy search (if filename contains base_name and "OD")
            if mask_path is None:
                for key in self.mask_map:
                    if base_name in key and "OD" in key:
                        mask_path = self.mask_map[key]
                        break

        image = Image.open(img_full_path).convert("RGB")
        
        # Load Mask if found, else Black Mask
        mask = Image.new('L', image.size, 0)
        try:
            if mask_path:
                mask = Image.open(mask_path).convert("L") # Grayscale
        except:
            pass

        # Resize
        image = TF.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        mask = TF.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.NEAREST)

        image = TF.to_tensor(image)
        mask = TF.to_tensor(mask)
        
        # Normalize Image
        image = TF.normalize(image, [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]) 
        
        # Threshold Mask (Drishti soft maps usually need > 0.1)
        mask[mask > 0.1] = 1.0
        mask[mask <= 0.1] = 0.0

        return image, mask

print("📂 Loading Datasets...")
try:
    print("--- Loading TRAIN Set ---")
    train_dataset = DrishtiDataset(image_dir=TRAIN_IMG_DIR, mask_dir=TRAIN_MASK_DIR)
    
    print("--- Loading TEST Set ---")
    test_dataset = DrishtiDataset(image_dir=TEST_IMG_DIR, mask_dir=TEST_MASK_DIR)
    
    if len(train_dataset) == 0 or len(test_dataset) == 0:
        print(f"❌ ERROR: Zero images found! Please check your paths.")
    else:
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        print(f"✅ Ready: {len(train_dataset)} Train images | {len(test_dataset)} Test images")

except Exception as e:
    print(f"❌ Dataset Error: {e}")

## 3. Model Architecture Factory
This section builds the chosen model (U-Net, Attention U-Net, or DeepLabV3).

In [ ]:
# --- U-NET & ATTENTION BLOCKS ---
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, 1, 0, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512], attention=False):
        super(UNet, self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.attention = attention
        self.att_blocks = nn.ModuleList() if attention else None
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))
            if self.attention: self.att_blocks.append(AttentionBlock(feature, feature, feature // 2))
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]
            if self.attention: skip_connection = self.att_blocks[idx//2](g=x, x=skip_connection)
            if x.shape != skip_connection.shape: x = TF.resize(x, size=skip_connection.shape[2:])
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)
        return self.final_conv(x)

# --- MODEL SELECTION LOGIC ---
print(f"🏗️ Building Model: {SEG_MODEL_NAME}...")

if SEG_MODEL_NAME == "DeepLabV3":
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    # Replace last layer for 1-channel output (Binary Segmentation)
    model.classifier[4] = nn.Conv2d(256, 1, kernel_size=(1, 1), stride=(1, 1))
    model.aux_classifier[4] = nn.Conv2d(256, 1, kernel_size=(1, 1), stride=(1, 1))
elif SEG_MODEL_NAME == "AttentionUNet":
    model = UNet(attention=True)
else: # Standard UNet
    model = UNet(attention=False)

model = model.to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')

## 4. Training Loop (Handles Model Differences)
DeepLab returns a dictionary `{'out': tensor}`, while U-Net returns a direct tensor. This loop handles both.

In [ ]:
def check_dice_score(loader, model, device="cuda"):
    dice_score = 0
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            
            output = model(x)
            if isinstance(output, dict): output = output['out']
            
            preds = torch.sigmoid(output)
            preds = (preds > 0.5).float()
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8)

    print(f"✅ Dice Score (on Test Set): {dice_score/len(loader):.4f}")
    model.train()
    return dice_score/len(loader)

train_losses = []
print("🔥 Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for data, targets in train_loader:
        data = data.to(DEVICE)
        targets = targets.float().to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            output = model(data)
            if isinstance(output, dict): output = output['out']
            loss = loss_fn(output, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(train_loader):.4f}")
    if (epoch + 1) % 5 == 0:
        check_dice_score(test_loader, model, device=DEVICE)

check_dice_score(test_loader, model, device=DEVICE) # Final check

## 5. Visualization
Saves the visual results for your paper.

In [ ]:
def save_predictions(loader, model, folder="segmentation_results/", device="cuda"):
    model.eval()
    if not os.path.exists(folder): os.makedirs(folder)
    
    for idx, (x, y) in enumerate(loader):
        x = x.to(device)
        with torch.no_grad():
            out = model(x)
            if isinstance(out, dict): out = out['out']
            preds = torch.sigmoid(out) > 0.5
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1,3,1); plt.title("Input"); plt.imshow(x[0].cpu().permute(1,2,0))
        plt.subplot(1,3,2); plt.title("Ground Truth"); plt.imshow(y[0].cpu().squeeze(), cmap='gray')
        plt.subplot(1,3,3); plt.title(f"{SEG_MODEL_NAME} Prediction"); plt.imshow(preds[0].cpu().squeeze(), cmap='gray')
        
        plt.savefig(f"{folder}{SEG_MODEL_NAME}_sample_{idx}.png")
        plt.show()
        if idx == 0: break

save_predictions(test_loader, model)
torch.save(model.state_dict(), f"{SEG_MODEL_NAME}_weights.pth")
print(f"✅ Saved model and images for {SEG_MODEL_NAME}")

## 6. Phase 6: Cross-Dataset Inference (Generalization Test)
Run this cell to test your trained model on a **new dataset** (e.g., your Hospital/RIM-ONE folders) to see visual predictions even if no masks exist.

In [ ]:
# === CONFIGURATION FOR NEW DATASET ===
# Update this to your 'Hospital Partition' or 'RIM-ONE' image folder
NEW_DATASET_DIR = r"path\to\Partitioned_by_Hospital\Test\Glaucoma"
# =====================================

print("🚀 Running Cross-Dataset Inference...")
try:
    # Load New Data (No masks needed, so we pass None)
    inference_dataset = DrishtiDataset(image_dir=NEW_DATASET_DIR, mask_dir=None)
    inference_loader = DataLoader(inference_dataset, batch_size=4, shuffle=True)
    
    model.eval()
    for idx, (x, y) in enumerate(inference_loader):
        x = x.to(DEVICE)
        with torch.no_grad():
            out = model(x)
            if isinstance(out, dict): out = out['out']
            preds = torch.sigmoid(out) > 0.5
        
        # Visualize
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.title("Input (New Dataset)")
        plt.imshow(x[0].cpu().permute(1, 2, 0))
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.title("AI Segmentation Prediction")
        plt.imshow(preds[0].cpu().squeeze(), cmap='jet')
        plt.axis('off')
        
        plt.tight_layout()
        plt.savefig(f"cross_dataset_result_{idx}.png")
        plt.show()
        
        if idx == 2: break # Show 3 examples
        
except Exception as e:
    print(f"❌ Inference Error: {e}")